# Portfolio Compliance Rule Engine

This project builds a simplified portfolio compliance workflow using Python and SQL.

The goal is to translate plain-English investment guidelines into automated rule logic, similar to how portfolio compliance systems monitor investment mandates.

Rules tested:
1. No single position greater than 5%
2. Maximum 20% exposure to the Technology sector
3. No bond holdings below BBB-

The project uses:
- Python for rule logic
- SQLite for SQL-based compliance checks
- Pandas for data handling
- CSV output for reporting

In [16]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [17]:
import pandas as pd
import sqlite3
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd

portfolio = pd.DataFrame({
    "ticker": ["AAPL", "MSFT", "NVDA", "JPM", "JNJ", "TSLA", "US10Y", "CORP_A", "CORP_B", "CASH"],
    "security_name": ["Apple Inc.", "Microsoft Corp.", "NVIDIA Corp.", "JPMorgan Chase", "Johnson & Johnson", "Tesla Inc.", "U.S. Treasury 10Y", "Investment Grade Corp Bond", "High Yield Corp Bond", "Cash"],
    "asset_class": ["Equity", "Equity", "Equity", "Equity", "Equity", "Equity", "Bond", "Bond", "Bond", "Cash"],
    "sector": ["Technology", "Technology", "Technology", "Financials", "Healthcare", "Consumer Discretionary", "Government", "Corporate", "Corporate", "Cash"],
    "credit_rating": ["AA+", "AAA", "AA-", "A+", "AAA", "BBB", "AAA", "A", "BB+", "AAA"],
    "weight_pct": [6.2, 5.8, 4.7, 4.5, 3.9, 5.3, 18.0, 14.0, 7.5, 30.1],
    "market_value": [62000, 58000, 47000, 45000, 39000, 53000, 180000, 140000, 75000, 301000]
})

portfolio.to_csv("mock_portfolio.csv", index=False)
portfolio

,ticker,security_name,asset_class,sector,credit_rating,weight_pct,market_value
0,AAPL,Apple Inc.,Equity,Technology,AA+,6.2,62000
1,MSFT,Microsoft Corp.,Equity,Technology,AAA,5.8,58000
2,NVDA,NVIDIA Corp.,Equity,Technology,AA-,4.7,47000
3,JPM,JPMorgan Chase,Equity,Financials,A+,4.5,45000
4,JNJ,Johnson & Johnson,Equity,Healthcare,AAA,3.9,39000
5,TSLA,Tesla Inc.,Equity,Consumer Discretionary,BBB,5.3,53000
6,US10Y,U.S. Treasury 10Y,Bond,Government,AAA,18.0,180000
7,CORP_A,Investment Grade Corp Bond,Bond,Corporate,A,14.0,140000
8,CORP_B,High Yield Corp Bond,Bond,Corporate,BB+,7.5,75000
9,CASH,Cash,Cash,Cash,AAA,30.1,301000


In [ ]:
def check_single_position_limit(df, max_weight=5.0):
    breaches = df[df["weight_pct"] > max_weight].copy()
    breaches["rule"] = f"No single position > {max_weight}%"
    breaches["violation_detail"] = breaches["ticker"] + " weight is " + breaches["weight_pct"].astype(str) + "%"
    return breaches


def check_sector_limit(df, sector="Technology", max_sector_weight=20.0):
    sector_weight = df.loc[df["sector"] == sector, "weight_pct"].sum()

    if sector_weight > max_sector_weight:
        return pd.DataFrame([{
            "ticker": "SECTOR_LEVEL",
            "security_name": sector,
            "asset_class": "Multiple",
            "sector": sector,
            "credit_rating": "N/A",
            "weight_pct": sector_weight,
            "market_value": df.loc[df["sector"] == sector, "market_value"].sum(),
            "rule": f"Max {max_sector_weight}% in {sector} sector",
            "violation_detail": f"{sector} exposure is {sector_weight:.2f}%"
        }])

    return pd.DataFrame()


def rating_rank(rating):
    rating_order = {
        "AAA": 1, "AA+": 2, "AA": 3, "AA-": 4,
        "A+": 5, "A": 6, "A-": 7,
        "BBB+": 8, "BBB": 9, "BBB-": 10,
        "BB+": 11, "BB": 12, "BB-": 13,
        "B+": 14, "B": 15, "B-": 16,
        "CCC": 17, "CC": 18, "C": 19, "D": 20
    }
    return rating_order.get(str(rating).upper(), 999)


def check_min_bond_rating(df, min_rating="BBB-"):
    threshold = rating_rank(min_rating)

    bonds = df[df["asset_class"] == "Bond"].copy()
    bonds["rating_score"] = bonds["credit_rating"].apply(rating_rank)

    breaches = bonds[bonds["rating_score"] > threshold].copy()
    breaches["rule"] = f"No bond holdings below {min_rating}"
    breaches["violation_detail"] = breaches["ticker"] + " is rated " + breaches["credit_rating"]

    return breaches.drop(columns=["rating_score"])

In [ ]:
python_violations = pd.concat([
    check_single_position_limit(portfolio),
    check_sector_limit(portfolio),
    check_min_bond_rating(portfolio)
], ignore_index=True)

python_violations[["rule", "ticker", "security_name", "weight_pct", "violation_detail"]]

,rule,ticker,security_name,weight_pct,violation_detail
0,No single position > 5.0%,AAPL,Apple Inc.,6.2,AAPL weight is 6.2%
1,No single position > 5.0%,MSFT,Microsoft Corp.,5.8,MSFT weight is 5.8%
2,No single position > 5.0%,TSLA,Tesla Inc.,5.3,TSLA weight is 5.3%
3,No single position > 5.0%,US10Y,U.S. Treasury 10Y,18.0,US10Y weight is 18.0%
4,No single position > 5.0%,CORP_A,Investment Grade Corp Bond,14.0,CORP_A weight is 14.0%
5,No single position > 5.0%,CORP_B,High Yield Corp Bond,7.5,CORP_B weight is 7.5%
6,No single position > 5.0%,CASH,Cash,30.1,CASH weight is 30.1%
7,No bond holdings below BBB-,CORP_B,High Yield Corp Bond,7.5,CORP_B is rated BB+


In [ ]:
import sqlite3

conn = sqlite3.connect("portfolio.db")
portfolio.to_sql("portfolio", conn, if_exists="replace", index=False)

10

In [ ]:
single_position_sql = """
SELECT
    ticker,
    security_name,
    asset_class,
    sector,
    credit_rating,
    weight_pct,
    market_value,
    'No single position > 5%' AS rule,
    ticker || ' weight is ' || weight_pct || '%' AS violation_detail
FROM portfolio
WHERE weight_pct > 5.0;
"""

sql_single_position = pd.read_sql_query(single_position_sql, conn)
sql_single_position

,ticker,security_name,asset_class,sector,credit_rating,weight_pct,market_value,rule,violation_detail
0,AAPL,Apple Inc.,Equity,Technology,AA+,6.2,62000,No single position > 5%,AAPL weight is 6.2%
1,MSFT,Microsoft Corp.,Equity,Technology,AAA,5.8,58000,No single position > 5%,MSFT weight is 5.8%
2,TSLA,Tesla Inc.,Equity,Consumer Discretionary,BBB,5.3,53000,No single position > 5%,TSLA weight is 5.3%
3,US10Y,U.S. Treasury 10Y,Bond,Government,AAA,18.0,180000,No single position > 5%,US10Y weight is 18.0%
4,CORP_A,Investment Grade Corp Bond,Bond,Corporate,A,14.0,140000,No single position > 5%,CORP_A weight is 14.0%
5,CORP_B,High Yield Corp Bond,Bond,Corporate,BB+,7.5,75000,No single position > 5%,CORP_B weight is 7.5%
6,CASH,Cash,Cash,Cash,AAA,30.1,301000,No single position > 5%,CASH weight is 30.1%


In [14]:
sector_sql = """
SELECT
    'SECTOR_LEVEL' AS ticker,
    'Technology' AS security_name,
    'Multiple' AS asset_class,
    sector,
    'N/A' AS credit_rating,
    SUM(weight_pct) AS weight_pct,
    SUM(market_value) AS market_value,
    'Max 20% in Technology sector' AS rule,
    'Technology exposure is ' || ROUND(SUM(weight_pct), 2) || '%' AS violation_detail
FROM portfolio
WHERE sector = 'Technology'
GROUP BY sector
HAVING SUM(weight_pct) > 20.0;
"""

sql_sector = pd.read_sql_query(sector_sql, conn)
sql_sector

,ticker,security_name,asset_class,sector,credit_rating,weight_pct,market_value,rule,violation_detail


In [ ]:
bond_rating_sql = """
SELECT
    ticker,
    security_name,
    asset_class,
    sector,
    credit_rating,
    weight_pct,
    market_value,
    'No bond holdings below BBB-' AS rule,
    ticker || ' is rated ' || credit_rating AS violation_detail
FROM portfolio
WHERE asset_class = 'Bond'
  AND credit_rating IN ('BB+', 'BB', 'BB-', 'B+', 'B', 'B-', 'CCC', 'CC', 'C', 'D');
"""

sql_bond_rating = pd.read_sql_query(bond_rating_sql, conn)
sql_bond_rating

,ticker,security_name,asset_class,sector,credit_rating,weight_pct,market_value,rule,violation_detail
0,CORP_B,High Yield Corp Bond,Bond,Corporate,BB+,7.5,75000,No bond holdings below BBB-,CORP_B is rated BB+


In [ ]:
python_violations["source"] = "Python"

sql_violations = pd.concat([
    sql_single_position,
    sql_sector,
    sql_bond_rating
], ignore_index=True)

sql_violations["source"] = "SQL"

final_report = pd.concat([python_violations, sql_violations], ignore_index=True)

final_report = final_report[[
    "source", "rule", "ticker", "security_name", "asset_class",
    "sector", "credit_rating", "weight_pct", "market_value", "violation_detail"
]]

final_report

/tmp/ipykernel_6118/3567555928.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  sql_violations = pd.concat([


,source,rule,ticker,security_name,asset_class,sector,credit_rating,weight_pct,market_value,violation_detail
0,Python,No single position > 5.0%,AAPL,Apple Inc.,Equity,Technology,AA+,6.2,62000,AAPL weight is 6.2%
1,Python,No single position > 5.0%,MSFT,Microsoft Corp.,Equity,Technology,AAA,5.8,58000,MSFT weight is 5.8%
2,Python,No single position > 5.0%,TSLA,Tesla Inc.,Equity,Consumer Discretionary,BBB,5.3,53000,TSLA weight is 5.3%
3,Python,No single position > 5.0%,US10Y,U.S. Treasury 10Y,Bond,Government,AAA,18.0,180000,US10Y weight is 18.0%
4,Python,No single position > 5.0%,CORP_A,Investment Grade Corp Bond,Bond,Corporate,A,14.0,140000,CORP_A weight is 14.0%
5,Python,No single position > 5.0%,CORP_B,High Yield Corp Bond,Bond,Corporate,BB+,7.5,75000,CORP_B weight is 7.5%
6,Python,No single position > 5.0%,CASH,Cash,Cash,Cash,AAA,30.1,301000,CASH weight is 30.1%
7,Python,No bond holdings below BBB-,CORP_B,High Yield Corp Bond,Bond,Corporate,BB+,7.5,75000,CORP_B is rated BB+
8,SQL,No single position > 5%,AAPL,Apple Inc.,Equity,Technology,AA+,6.2,62000,AAPL weight is 6.2%
9,SQL,No single position > 5%,MSFT,Microsoft Corp.,Equity,Technology,AAA,5.8,58000,MSFT weight is 5.8%


In [ ]:
final_report.to_csv("compliance_violations_report.csv", index=False)

In [ ]:
from google.colab import files
files.download("compliance_violations_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Portfolio Compliance Assessment

To make the compliance results easier to interpret, the portfolio is displayed with an automated compliance status for each holding.

- ✅ Compliant holdings satisfy all investment guidelines.
- ❌ Violating holdings breach one or more investment rules and are highlighted for review.

In [15]:
# Create a set of violating tickers
violating_tickers = set(final_report["ticker"])

# Add compliance status
portfolio["Compliance Status"] = portfolio["ticker"].apply(
    lambda x: "❌ VIOLATION" if x in violating_tickers else "✅ COMPLIANT"
)

# Highlight rows
def highlight(row):
    if row["Compliance Status"] == "❌ VIOLATION":
        return ['background-color:#ffcccc; color:darkred; font-weight:bold'] * len(row)
    else:
        return ['background-color:#d9f7d9; color:darkgreen'] * len(row)

portfolio.style.apply(highlight, axis=1)

,ticker,security_name,asset_class,sector,credit_rating,weight_pct,market_value,Compliance Status
0,AAPL,Apple Inc.,Equity,Technology,AA+,6.200000,62000,❌ VIOLATION
1,MSFT,Microsoft Corp.,Equity,Technology,AAA,5.800000,58000,❌ VIOLATION
2,NVDA,NVIDIA Corp.,Equity,Technology,AA-,4.700000,47000,✅ COMPLIANT
3,JPM,JPMorgan Chase,Equity,Financials,A+,4.500000,45000,✅ COMPLIANT
4,JNJ,Johnson & Johnson,Equity,Healthcare,AAA,3.900000,39000,✅ COMPLIANT
5,TSLA,Tesla Inc.,Equity,Consumer Discretionary,BBB,5.300000,53000,❌ VIOLATION
6,US10Y,U.S. Treasury 10Y,Bond,Government,AAA,18.000000,180000,❌ VIOLATION
7,CORP_A,Investment Grade Corp Bond,Bond,Corporate,A,14.000000,140000,❌ VIOLATION
8,CORP_B,High Yield Corp Bond,Bond,Corporate,BB+,7.500000,75000,❌ VIOLATION
9,CASH,Cash,Cash,Cash,AAA,30.100000,301000,❌ VIOLATION
